# Session 3 — Functions, Scope & Recursion

> params, *args/**kwargs, scope, the mutable-default bug; base/recursive case, the call stack, nested data.

**How to use this notebook:** read each cell, **predict** what it prints, then run it with **Shift + Enter**. Change one thing and predict again — the surprise is the lesson. Practice tasks (with collapsed solutions) are at the bottom.

**Tips:** press **Tab** to autocomplete a name, and **Shift + Tab** for a function's help. Need a library? Run `%pip install <name>` in a cell (e.g. `%pip install pandas`) — in the browser (JupyterLite) that fetches a Pyodide build and lasts for the session.

In [ ]:
# ======================================================================
# PART A — Functions, Scope & Reusability

In [ ]:
# ======================================================================

import functools

In [ ]:
# --- 1. return vs print --------------------------------------------------
def avg(xs: list[float]) -> float:
    """Return the mean of xs."""
    return sum(xs) / len(xs)

def show(xs):
    print("mean is", sum(xs) / len(xs))

x = avg([1, 2, 3])      # 2.0  (a value we can keep using)
y = show([1, 2, 3])     # prints, but...
print("x =", x, "| y =", y)     # y is None!

In [ ]:
# --- 2. defaults, *args, **kwargs ---------------------------------------
def grade(score, scale=100, passing=60):
    pct = score / scale
    return "PASS" if score >= passing else "FAIL", round(pct, 3)

print(grade(85), grade(40, passing=35))
print("via **dict:", grade(**{"score": 85, "passing": 80}))  # ** unpacks a dict into kwargs

def total(*args):              # collect positionals into a tuple
    return sum(args)
print("total:", total(1, 2, 3, 4))

def tag(**kwargs):             # collect keywords into a dict
    return kwargs
print("tag:", tag(name="Ana", gpa=3.9))

scores = [91, 58, 73]
print("unpacked into total:", total(*scores))   # * unpacks the list

In [ ]:
# --- 3. Keyword-only arguments (everything after * must be named) -------
def report(name, *, verbose=False):    # `verbose` can't be passed positionally
    return f"{name} (full report)" if verbose else name
print("\n", report("Ana", verbose=True))
# report("Ana", True)   # would raise TypeError — that's the point: clarity at the call site

In [ ]:
# --- 4. A decorator: a function that wraps another function -------------
def announce(func):
    @functools.wraps(func)            # keep func's name/docstring on the wrapper
    def wrapper(*args, **kwargs):     # *args/**kwargs forward ANY signature
        print(f"  calling {func.__name__}{args}")
        return func(*args, **kwargs)
    return wrapper

@announce                              # sugar for:  add = announce(add)
def add(a, b):
    return a + b

print("\ndecorated:", add(2, 3))

In [ ]:
# --- 5. Closures + nonlocal: a function that remembers state ------------
def make_counter():
    count = 0
    def step():
        nonlocal count                # rebind the enclosing variable, not a new local
        count += 1
        return count
    return step

tick = make_counter()
print("\ncounter:", tick(), tick(), tick())   # 1 2 3

In [ ]:
# --- 6. TRAP: mutable default argument ----------------------------------
def add_bad(name, roster=[]):       # ❌ shared default
    roster.append(name)
    return roster

print("\nBUGGY:")
print(add_bad("Ana"))               # ['Ana']
print(add_bad("Ben"))               # ['Ana', 'Ben']  <- persists!

def add_ok(name, roster=None):      # ✅
    if roster is None:
        roster = []
    roster.append(name)
    return roster

print("FIXED:")
print(add_ok("Ana"))                # ['Ana']
print(add_ok("Ben"))                # ['Ben']  <- fresh each call

In [ ]:
# --- 7. Scope: UnboundLocalError demo (commented) -----------------------
# count = 0
# def bump():
#     count = count + 1   # UnboundLocalError: assigning makes `count` local
# Prefer returning a value (or use `nonlocal`/a closure, as in block 5).

In [ ]:
# ======================================================================
# PART B — Recursion & Recursive Thinking

In [ ]:
# ======================================================================

import sys
import functools

In [ ]:
# 1) The shape of EVERY recursive function: a base case + a recursive case.
def countdown(n):
    if n <= 0:                 # BASE CASE — the stop condition
        print("liftoff!")
        return
    print(n)
    countdown(n - 1)           # RECURSIVE CASE — same problem, smaller input

print("countdown:")
countdown(3)

In [ ]:
# 2) Recursion vs iteration: same answer, two styles. Each call adds a stack frame.
def factorial(n):
    if n <= 1:
        return 1               # base case
    return n * factorial(n - 1)   # remember to RETURN the recursive call

def factorial_loop(n):
    total = 1
    for k in range(2, n + 1):
        total *= k
    return total

print("\nfactorial(5):", factorial(5), "==", factorial_loop(5))

In [ ]:
# 3) Memoization: @lru_cache remembers past calls so each input is computed once.
#    Naive fibonacci recomputes the same values exponentially; the cache makes it linear.
calls = 0
def fib_naive(n):
    global calls
    calls += 1
    return n if n < 2 else fib_naive(n - 1) + fib_naive(n - 2)

@functools.lru_cache(maxsize=None)     # one decorator turns the slow version fast
def fib_fast(n):
    return n if n < 2 else fib_fast(n - 1) + fib_fast(n - 2)

print("\nfib_naive(30):", fib_naive(30), "in", calls, "calls")
print("fib_fast(30): ", fib_fast(30), "->", fib_fast.cache_info())   # far fewer hits

In [ ]:
# 4) Mutual recursion: two functions that call each other toward a shared base case.
def is_even(n):
    return True if n == 0 else is_odd(n - 1)
def is_odd(n):
    return False if n == 0 else is_even(n - 1)
print("\nis_even(10):", is_even(10), "| is_odd(7):", is_odd(7))

In [ ]:
# 5) Where recursion SHINES: naturally NESTED data, where one loop can't reach
#    all the way down. This is shaped like a real nested-JSON export.
export = {
    "cohort": "2026",
    "students": [
        {"name": "Ana", "scores": [91, 88]},
        {"name": "Ben", "scores": [58, [60, 64]]},   # arbitrarily nested
    ],
}

def deep_sum(obj):
    """Add up every number found anywhere inside nested lists/dicts."""
    if isinstance(obj, bool):                 # bool is an int subclass (Session 2!)
        return 0
    if isinstance(obj, (int, float)):
        return obj
    if isinstance(obj, dict):
        return sum(deep_sum(v) for v in obj.values())
    if isinstance(obj, (list, tuple)):
        return sum(deep_sum(x) for x in obj)
    return 0                                  # strings, None, etc. contribute nothing

print("\ndeep_sum of nested export:", deep_sum(export))   # 91+88+58+60+64 = 361

In [ ]:
# 6) The trap: with no reachable base case, recursion never stops. Python has no
#    tail-call optimization, so it just piles up stack frames until it gives up.
print("\nPython's recursion limit:", sys.getrecursionlimit())

def runaway(n):
    return runaway(n + 1)        # BUG: never reaches a base case

try:
    runaway(0)
except RecursionError:
    print("RecursionError: maximum recursion depth exceeded (as expected)")

## Now you try — practice

# Session 3 — Practice: Functions & Recursion

This 2-hour session has two halves. Do **Part A** after the first topic, **Part B** after the second. Predict every output before you run it.

## Part A — Functions, Scope & Reusability

### Task 1 — Grade-functions module
Write three functions with docstrings and type hints:
- `class_average(scores: list[float]) -> float`
- `letter_grade(score: float) -> str`  (reuse Session 3)
- `pass_rate(scores: list[float], passing: float = 60) -> float`  (fraction passing, 0–1)

Use bool-summing for `pass_rate` (recall `sum(s >= passing for s in scores)`).

### Task 2 — Reproduce & fix the mutable-default bug
Write `add_note(text, notes=[])` that appends and returns. Call it three times and watch
the list grow. Then fix it with the `None` pattern and prove each call starts fresh.

### Task 3 — *args summary
Write `summary(*scores)` that returns a dict `{"n":..., "mean":..., "max":..., "min":...}`.
Call it both as `summary(91, 58, 73)` and as `summary(*my_list)`.

### Bonus — Pythonic idiom drill
Cover the `# ->` answers, predict each line, then run.

```python
def f(a, *, b):          # everything after * is keyword-only
    return a, b
print(f(1, b=2))                     # -> (1, 2)
print(f(**{"a": 1, "b": 9}))         # -> (1, 9)   (** unpacks a dict into arguments)
```

## Part B — Recursion & Recursive Thinking

### Task 1 — Recursive sum
Write `rsum(n)` that adds `1 + 2 + ... + n` **with recursion** (no loop).
Name the base case out loud before you write it. Test `rsum(5)` and `rsum(0)`.

### Task 2 — Recursion vs iteration
Write `reverse(s)` that reverses a string recursively. Then write the loop version.
Which reads more clearly to you? Test `reverse("data")`.

### Task 3 — Flatten nested data
Write `flatten(xs)` that turns a list-of-lists (nested to any depth) into one flat list:
`flatten([1, [2, [3, 4]], 5])` → `[1, 2, 3, 4, 5]`. This is the move for nested JSON/exports.

### Task 4 — How deep does it go?
Write `depth(xs)` returning how deeply a list is nested:
`depth([1, [2, [3, [4]]]])` → `4`, `depth([1, 2, 3])` → `1`, `depth(5)` → `0`.

### Task 5 — Trap check
1. Why does this raise `RecursionError`, and what's the fix?
   ```python
   def f(n):
       return n + f(n - 1)
   ```
2. This returns `None` instead of a number — why?
   ```python
   def fact(n):
       if n <= 1:
           return 1
       n * fact(n - 1)
   ```
3. Name one case where a plain loop is the better choice over recursion.

### Bonus — Pythonic idiom drill
One decorator makes exponential recursion instant by remembering past calls.

```python
import functools

@functools.cache                     # memoize: each n is computed once
def fib(n):
    return n if n < 2 else fib(n - 1) + fib(n - 2)
print(fib(35))                       # -> 9227465   (try this WITHOUT @cache... then wait)
```

---

In [ ]:
# Your practice work — type here. Predict before you run.


<details>
<summary><strong>Show solutions</strong></summary>

### Part A — Functions, Scope & Reusability

```python
def class_average(scores: list[float]) -> float:
    """Mean of scores."""
    return sum(scores) / len(scores)

def letter_grade(score: float) -> str:
    """A/B/C/D/F by 90/80/70/60 cutoffs."""
    for cutoff, letter in [(90,"A"),(80,"B"),(70,"C"),(60,"D")]:
        if score >= cutoff:
            return letter
    return "F"

def pass_rate(scores: list[float], passing: float = 60) -> float:
    """Fraction of scores >= passing (0..1)."""
    return sum(s >= passing for s in scores) / len(scores)

# Task 2
def add_note(text, notes=None):     # fixed version
    if notes is None:
        notes = []
    notes.append(text)
    return notes

# Task 3
def summary(*scores):
    return {"n": len(scores), "mean": sum(scores)/len(scores),
            "max": max(scores), "min": min(scores)}
print(summary(91, 58, 73))
print(summary(*[91, 58, 73]))
```

### Part B — Recursion & Recursive Thinking

```python
# 1
def rsum(n):
    if n == 0:                      # base case
        return 0
    return n + rsum(n - 1)
print(rsum(5), rsum(0))             # 15 0

# 2
def reverse(s):
    if s == "":                     # base case: empty string
        return ""
    return reverse(s[1:]) + s[0]    # all-but-first, reversed, then first
print(reverse("data"))             # "atad"
# loop version: "".join(reversed(s))  — usually clearer for flat strings

# 3
def flatten(xs):
    out = []
    for x in xs:
        if isinstance(x, list):
            out.extend(flatten(x))  # recurse into the sub-list
        else:
            out.append(x)
    return out
print(flatten([1, [2, [3, 4]], 5]))   # [1, 2, 3, 4, 5]

# 4
def depth(xs):
    if not isinstance(xs, list):
        return 0                              # a non-list has no nesting
    return 1 + max((depth(x) for x in xs), default=0)
print(depth([1, [2, [3, [4]]]]), depth([1, 2, 3]), depth(5))   # 4 1 0

# 5
# 1) No reachable base case -> the calls never stop -> stack overflows.
#    Fix: add `if n == 0: return 0` (or n <= 0) at the top.
# 2) The recursive case computes n*fact(n-1) but never RETURNs it,
#    so the function falls off the end and returns None. Add `return`.
# 3) A loop is better when the work is a simple flat sequence, or when the
#    depth could exceed ~1000 (Python has no tail-call optimization, so deep
#    recursion hits RecursionError where a loop would be fine).
```

</details>